<a href="https://colab.research.google.com/github/chloemjshin/2026-1-archive/blob/main/bic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import files
uploaded = files.upload()

Saving bichon.zip to bichon.zip


In [7]:
# ==========================================
# 0. 라이브러리 세팅
# ==========================================
!pip install -U ultralytics tqdm opencv-python-headless

import os, zipfile, cv2, random, shutil
import numpy as np
from tqdm.auto import tqdm
from ultralytics import YOLO

# ==========================================
# 1. 경로 설정 및 압축 해제
# ==========================================
ZIP_PATH = '/content/bichon.zip'
RAW_DIR = '/content/raw_extracted'
FINAL_YOLO_DIR = '/content/YOLO_DATASET'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError("bichon.zip 업로드 안 됨")

if os.path.exists(RAW_DIR): shutil.rmtree(RAW_DIR)
os.makedirs(RAW_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(RAW_DIR)

# ==========================================
# 2. 이미지 스캔
# ==========================================
bg_paths, obj_paths = [], []

for root, _, files in os.walk(RAW_DIR):
    if '__MACOSX' in root: continue
    for f in files:
        if not f.lower().endswith(('.png','.jpg')): continue
        p = os.path.join(root, f)
        if 'background' in root.lower():
            bg_paths.append(p)
        elif 'object' in root.lower():
            obj_paths.append(p)

print(f"background: {len(bg_paths)}, bichon: {len(obj_paths)}")

# ==========================================
# 3. YOLO 폴더 구조
# ==========================================
for split in ['train','val']:
    os.makedirs(f'{FINAL_YOLO_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{FINAL_YOLO_DIR}/labels/{split}', exist_ok=True)

CLASS_ID = 0
TOTAL_IMAGES = 800
VAL_RATIO = 0.2

# ==========================================
# 4. 유틸 함수
# ==========================================
def overlay(bg, fg, x, y):
    h, w = fg.shape[:2]
    alpha = fg[:,:,3:] / 255.0
    roi = bg[y:y+h, x:x+w]
    bg[y:y+h, x:x+w] = fg[:,:,:3]*alpha + roi*(1-alpha)
    return bg

def rotate(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2,h//2), angle, 1)
    return cv2.warpAffine(img, M, (w,h),
                           borderMode=cv2.BORDER_CONSTANT,
                           borderValue=(0,0,0,0))

def fake_bichon(img):
    h,w = img.shape[:2]
    for _ in range(random.randint(3,6)):
        cv2.circle(img,
                   (random.randint(0,w), random.randint(0,h)),
                   random.randint(3,8),
                   (20,20,20,255),
                   -1)
    img[:,:,:3] = cv2.GaussianBlur(img[:,:,:3], (7,7), 0)
    return img

def random_color_aug(img):
    img = img.astype(np.float32)
    img *= random.uniform(0.7,1.3)
    img = np.clip(img,0,255)
    return img.astype(np.uint8)

# ==========================================
# 5. 데이터 생성
# ==========================================
print("🚀 데이터 생성 중...")

for i in tqdm(range(TOTAL_IMAGES)):
    split = 'val' if random.random() < VAL_RATIO else 'train'
    canvas = np.full((640,640,3), 240, dtype=np.uint8)
    labels = []

    # 배경 clutter
    for _ in range(random.randint(15,30)):
        img = cv2.imread(random.choice(bg_paths), cv2.IMREAD_UNCHANGED)
        if img is None: continue
        img = cv2.resize(img, (0,0),
                         fx=random.uniform(0.1,0.25),
                         fy=random.uniform(0.1,0.25))
        if random.random()<0.3:
            img = fake_bichon(img)
        img = rotate(img, random.randint(0,360))
        x = random.randint(0,640-img.shape[1]-1)
        y = random.randint(0,640-img.shape[0]-1)
        canvas = overlay(canvas, img, x, y)

    # 진짜 비숑 (1~3마리)
    for _ in range(random.randint(1,3)):
        img = cv2.imread(random.choice(obj_paths), cv2.IMREAD_UNCHANGED)
        if img is None: continue
        img = cv2.resize(img, (0,0),
                         fx=random.uniform(0.12,0.22),
                         fy=random.uniform(0.12,0.22))
        img = rotate(img, random.randint(-20,20))
        x = random.randint(0,640-img.shape[1]-1)
        y = random.randint(0,640-img.shape[0]-1)
        canvas = overlay(canvas, img, x, y)

        cx = (x+img.shape[1]/2)/640
        cy = (y+img.shape[0]/2)/640
        w  = img.shape[1]/640
        h  = img.shape[0]/640
        labels.append(f"{CLASS_ID} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    canvas = random_color_aug(canvas)

    fname = f'bichon_{i:04d}'
    cv2.imwrite(f'{FINAL_YOLO_DIR}/images/{split}/{fname}.jpg', canvas)
    with open(f'{FINAL_YOLO_DIR}/labels/{split}/{fname}.txt','w') as f:
        f.write('\n'.join(labels))

print("✅ 데이터셋 완성")

# ==========================================
# 6. YAML
# ==========================================
yaml_text = f"""
path: {FINAL_YOLO_DIR}
train: images/train
val: images/val
names:
  0: bichon
"""
with open('/content/bichon.yaml','w') as f:
    f.write(yaml_text)

# ==========================================
# 7. 학습
# ==========================================
print("🔥 YOLO 학습 시작")
model = YOLO('yolo11s.pt')
model.train(
    data='/content/bichon.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project='bichon_camo_v11',
    name='train_run',
    exist_ok=True
)

print("🎯 완료! weights/best.pt 확인")

background: 4, bichon: 10
🚀 데이터 생성 중...


  0%|          | 0/800 [00:00<?, ?it/s]

✅ 데이터셋 완성
🔥 YOLO 학습 시작
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/bichon.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True